In [1]:
import asyncio
import pandas as pd
from datetime import datetime, timedelta
from ib_insync import IB, util, Order, Forex, Contract
import time

# --- 1. 全局配置 ---
TWS_HOST = '127.0.0.1'
TWS_PORT = 7497  # 模拟盘通常为 7497，实盘为 7492
CLIENT_ID = 0    # 必须为 0 以认领外部订单

# 品种规则：[最大手数, 止盈点数, 止损点数, 价格精度]
SYMBOL_CONFIG = {
    'MES': {
        'max_qty': 2, 'tp': 20, 'sl': 20, 'precision': 2,
        'loss_threshold_base': 10  # 亏损超过 10 USD 才计入连续止损
    },
    'N225MC': {
        'max_qty': 6, 'tp': 100, 'sl': 100, 'precision': 0,
        'loss_threshold_base': 5   # 亏损折合超过 5 USD 才计入
    }
}
global_dll_lock_time = timedelta(hours=24)
symbol_slc_lock_time = timedelta(minutes=15)

# --- 2. 风控规则引擎 ---

class RiskRule:
    def check(self, trade, fills_df):
        return {"is_allowed": True, "reason": ""}

class SymbolAndPositionRule(RiskRule):
    """规则 1 & 4: 品种白名单与仓位限制"""
    def check(self, trade, fills_df):
        sym = trade.contract.symbol
        qty = trade.order.totalQuantity
        if sym not in SYMBOL_CONFIG:
            return {"is_allowed": False, "reason": f"品种 {sym} 不在白名单"}
        if qty > SYMBOL_CONFIG[sym]['max_qty']:
            return {"is_allowed": False, "reason": f"{sym} 超过限额 {SYMBOL_CONFIG[sym]['max_qty']}"}
        return {"is_allowed": True}

class CooldownAndDailyLossRule(RiskRule):
    """规则 2, 3 & 新增: 连续止损/日损锁定"""
    def __init__(self, ib, base_ccy='USD'):
        self.ib = ib
        self.base_ccy = base_ccy
        self.lock_until = {} # 品种锁定: {symbol: time}
        self.cooldown_active = {} # 标记是否正在执行“15分钟冷静期”任务
        self.retry_counts = {}     # 记录每个品种今日已豁免重试的次数
        self.fx_cache = {}   # 汇率缓存: {ccy: (rate, expire_time)}
        self.fx_last_update = 0
        self.global_lock_until = None

    # async def update_fx_rates(self):
    #     """最强兼容性汇率更新：显式定义 CASH 合约"""
    #     now = time.time()
    #     if now - self.fx_last_update < 3600: return 
    #     try:
    #         # 使用基础 Contract 类定义，直接避开 Forex 类的潜在歧义
    #         contract = Contract()
    #         contract.symbol = 'JPY'
    #         contract.secType = 'CASH'
    #         contract.exchange = 'IDEALPRO'
    #         contract.currency = self.base_ccy  # 'USD'

    #         # 1. 先验证合约
    #         qualified = await self.ib.qualifyContractsAsync(contract)
    #         if not qualified:
    #             print(f"⚠️ 无法验证 JPY.USD 合约，请检查 TWS 是否有外汇行情权限")
    #             # 如果验证失败，使用一个硬编码的参考值防止报错
    #             self.fx_cache['JPY'] = 0.0066 
    #             return

    #         # 2. 请求快照行情 (snapshot=True 消耗更少资源且更稳定)
    #         tickers = await self.ib.reqTickersAsync(qualified[0])
    #         if tickers:
    #             # 尝试获取中点价，如果没有则取买卖价平均值
    #             t = tickers[0]
    #             rate = t.midpoint()
    #             if not (rate > 0): # 如果 midpoint 无效，尝试用 bid/ask
    #                 if t.bid > 0 and t.ask > 0:
    #                     rate = (t.bid + t.ask) / 2
                
    #             if rate > 0 and rate == rate:
    #                 self.fx_cache['JPY'] = rate
    #                 self.fx_last_update = now
    #                 print(f"🔄 [系统汇率刷新成功] 1 JPY = {rate} USD")
    #             else:
    #                 print("⚠️ 汇率获取结果无效，请确认 TWS 市场数据已连接")
    #     except Exception as e:
    #         print(f"⚠️ 汇率刷新失败 (可能是权限或连接问题): {e}")

    def get_rate(self, ccy):
        if ccy == self.base_ccy: return 1.0
        # 如果缓存没有，给一个保守估计值，等后台任务更新
        return self.fx_cache.get(ccy, 0.0066 if ccy == 'JPY' else 1.0)
    
    def get_ny_date(self):
        """获取当前的纽约日期字符串"""
         # 获取当前纽约时间
        return pd.Timestamp.now(tz='US/Eastern').date().strftime('%Y-%m-%d');
        
    def check(self, trade, fills_df):
        now = datetime.now()
        sym = trade.contract.symbol

        # 1. 检查全局/品种锁定
        if self.global_lock_until and now < self.global_lock_until:
            return {"is_allowed": False, "reason": f"全局锁定中至 {self.global_lock_until}"}
        if sym in self.lock_until and now < self.lock_until[sym]:
            return {"is_allowed": False, "reason": f"{sym} 处于冷却期至 {self.lock_until[sym]}"}

        if fills_df.empty: return {"is_allowed": True}

        # 2. 计算多币种折算 PnL (24小时)
        day_limit = now - timedelta(hours=24)
        recent = fills_df[fills_df['ny_time'] >= day_limit].copy()
        
        # 计算折算后的 PnL 并存入新列
        recent['pnl_base'] = recent.apply(
            lambda x: x['pnl'] * self.get_rate(x['currency']), axis=1
        )
        # 把打平附近的交易的剔除
        valid_symbol_trades = recent[
                (recent['symbol'] == sym) & 
                (recent['pnl_base'] <= -SYMBOL_CONFIG[sym]['loss_threshold_base']) &
                (recent['pnl_base'] >=  SYMBOL_CONFIG[sym]['loss_threshold_base']) 
            ]

        # 3. 计算全局日损 (不受门槛限制，所有亏损都计入，也不区分品种，全部计入)
        total_pnl_base = recent['pnl_base'].sum()

        # A. 全局日损规则 (>200 or loss 3 锁 24h)
        last_3 = valid_symbol_trades.head(3)['pnl'].tolist()
        last_3_loss = (len(last_3) == 3 and all(p < 0 for p in last_3))
        if total_pnl_base <= -200 or last_3_loss:
            self.global_lock_until = now + global_dll_lock_time
            return {"is_allowed": False, "reason": f"全局日损 {total_pnl:.2f} or 连损次数 超限，锁24h"}
        
        # B. 短期锁定规则 (日损>100 或 连续2笔止损，锁15min)
        last_2 = valid_symbol_trades.head(2)['pnl'].tolist()
        last_2_loss = (len(last_2) == 2 and all(p < 0 for p in last_2))
        ny_date = self.get_ny_date()
        retry_key = f"{sym}_{ny_date}"
        
        if total_pnl_base <= -100 or last_2_loss:

            used_tries = self.retry_counts.get(retry_key, 0)
            if used_tries < 2:
                # 只有在当前没有活跃的冷却记录时才触发，防止重入
                if sym not in self.cooldown_active or self.cooldown_active[sym] is False:
                    self.lock_until[sym] = now + symbol_slc_lock_time
                    self.cooldown_active[sym] = True # 打开冷却后重试能力
                    reason = "日损超100" if total_pnl_base <= -100 else "连续2次止损"
                    return {"is_allowed": False, "reason": f"触发{reason}，锁定15min"}        
    
                if sym in self.cooldown_active and self.cooldown_active[sym] is True:
                    # 这一单我们放行，但立即把开关关掉，关闭冷却后重试能力
                    # 这样如果这单没成交或被撤单，下一单由于逻辑 3 依然会被重新判定
                    self.cooldown_active[sym] = False 
                    self.retry_counts[retry_key] = used_tries + 1
                    print(f"🔔 {sym} 冷静期已过，允许尝试一单进行回血，今日尝试次数:{self.retry_counts[retry_key]}")
                    return {"is_allowed": True}
        
        # C. 通过所有规则
        return {"is_allowed": True}

In [2]:
# --- 3. 核心管理器 ---

class RiskManager:
    def __init__(self, ib):
        self.ib = ib
        self.processed_ids = set()
        self.bracket_applied = set()
        self.rules = [
            SymbolAndPositionRule(),
            CooldownAndDailyLossRule(ib)
        ]
        self.cooldown_rule = CooldownAndDailyLossRule(ib)

    def get_fills(self):
        fills = self.ib.fills()
        if not fills: return pd.DataFrame()
        df = util.df(fills)
        df['symbol'] = df['contract'].apply(lambda x: x.symbol)
        df['currency'] = df['contract'].apply(lambda x: x.currency)
        df['pnl'] = df['commissionReport'].apply(lambda x: x.realizedPNL)
        df['ny_time'] = pd.to_datetime(df['execution'].apply(lambda x: x.time)).dt.tz_convert('US/Eastern').dt.tz_localize(None)
        return df[df['pnl'] != 0].sort_values('ny_time', ascending=False)

    def onOpenOrder(self, trade):
        p_id = trade.order.permId
        if p_id in self.processed_ids: return
        if trade.orderStatus.status not in ['PreSubmitted', 'Submitted']: return

        # 执行拦截逻辑
        fills_df = self.get_fills()
        for rule in self.rules:
            res = rule.check(trade, fills_df)
            if not res['is_allowed']:
                print(f"🛑 拦截: {res['reason']}")
                self.processed_ids.add(p_id)
                asyncio.create_task(self.cancel_order(p_id))
                return

        print(f"✅ 通过: {trade.contract.symbol} {p_id}")
        self.processed_ids.add(p_id)

        # # 2. 检查持仓：只有在【无仓位】时才触发自动补单
        # # 获取当前所有持仓
        # current_positions = self.ib.positions()
        # # 找到当前品种的持仓数值
        # pos_qty = next((p.position for p in current_positions if p.contract.symbol == sym), 0)

        # if trade.order.parentId == 0:
        #     if pos_qty == 0:
        #         print(f"🆕 {trade.contract.symbol} 当前无持仓，准备自动补齐止盈止损...")
        #         # 尝试自动补止盈止损
        #         self.auto_bracket(trade)
        #     else:
        #         print(f"⚠️ {trade.contract.symbol} 已有持仓 ({pos_qty})，跳过自动补单逻辑以防冲突。")
        
    async def cancel_order(self, p_id):
        # self.ib.reqAutoOpenOrders(True)
        # await asyncio.sleep(0.5)
        for o in self.ib.reqOpenOrders():
            if o.order.permId == p_id and o.order.orderId > 0:
                self.ib.cancelOrder(o.order)

    def auto_bracket(self, trade):
        sym = trade.contract.symbol
        p_id = trade.order.permId
        if sym in SYMBOL_CONFIG and trade.order.parentId == 0 and p_id not in self.bracket_applied:
            asyncio.create_task(self.place_bracket(trade))

    async def place_bracket(self, parent_trade):
        # self.ib.reqAutoOpenOrders(True)
        # 增加一点点等待时间，确保 TWS 已经处理完父订单并分配了有效 OrderId
        # await asyncio.sleep(1.2)
        
        p_id = parent_trade.order.permId
        trades = self.ib.reqOpenOrders()
        
        # 寻找对应的父订单对象
        parent = next((t for t in trades if t.order.permId == p_id and t.order.orderId > 0), None)
        
        # 如果找不到父订单，或者已经有子订单存在了，则退出
        if not parent or any(o.parentId == parent.order.orderId for o in self.ib.openOrders()): 
            return

        symbol = parent.contract.symbol
        cfg = SYMBOL_CONFIG[symbol]
        
        # 获取最新的行情快照
        tickers = await self.ib.reqTickersAsync(parent.contract)
        if not tickers:
            print(f"⚠️ {sym} 无法获取行情数据")
            return
        ticker = tickers[0]
        # 获取中点价 (midpoint)
        price = parent.order.lmtPrice or ticker.midpoint()
        
        if not price or price != price: 
            print(f"⚠️ {symbol} 无法获取有效参考价，补单取消")
            return

        is_buy = parent.order.action == 'BUY'
        # 使用 p 后缀区分临时变量
        tp_p = price + (cfg['tp'] if is_buy else -cfg['tp'])
        sl_p = price - (cfg['sl'] if is_buy else -cfg['sl'])
        
        opp = 'SELL' if is_buy else 'BUY'
        prec = cfg['precision']
        
        # 格式化价格：日经(NK225MC)取整，标普(MES)保留2位
        final_tp = round(tp_p, prec) if prec > 0 else int(tp_p)
        final_sl = round(sl_p, prec) if prec > 0 else int(sl_p)
        
        # 构造止损
        sl_order = Order(action=opp, totalQuantity=parent.order.totalQuantity, orderType='STP', 
                         auxPrice=final_sl, parentId=parent.order.orderId, transmit=True)

        # 构造止盈
        tp_order = Order(action=opp, totalQuantity=parent.order.totalQuantity, orderType='LMT', 
                         lmtPrice=final_tp, parentId=parent.order.orderId, transmit=True)
        
        self.ib.placeOrder(parent.contract, sl_order)
        self.ib.placeOrder(parent.contract, tp_order)
        
        # 记录已处理，防止重复
        self.bracket_applied.add(p_id)
        print(f"🎯 自动补单成功: {symbol} TP@{final_tp}, SL@{final_sl}")

# --- 4. 主程序运行 ---

async def main():
    ib = IB()
    mgr = RiskManager(ib)
    ib.openOrderEvent += mgr.onOpenOrder
    while True:
        try:
            if not ib.isConnected():
                await ib.connectAsync(TWS_HOST, TWS_PORT, clientId=CLIENT_ID)
                ib.reqAutoOpenOrders(True)
                print("🌐 连接已建立，风控监控中...")
            # 异步更新汇率
            # await mgr.cooldown_rule.update_fx_rates()
            await asyncio.sleep(10)
        except Exception as e:
            print(f"⚠️ 连接错误: {e}")
            await asyncio.sleep(10)

if __name__ == "__main__":
    # 启动异步循环
    util.patchAsyncio()
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("👋 系统关闭")

🌐 连接已建立，风控监控中...


Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u5df2\u8fde\u63a5\uff1a usfuture; jfarm; usfarm; fundfarm; ushmds; secdefil. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u672a\u8fde\u63a5\uff1a euhmds.


✅ 通过: MES 1935278949
✅ 通过: MES 1935278950
✅ 通过: MES 1935278951
✅ 通过: MES 1935279694
✅ 通过: MES 1935279695
✅ 通过: MES 1935279696
✅ 通过: MES 1935279885
✅ 通过: MES 1935279886
✅ 通过: MES 1935279887
✅ 通过: MES 1935279889
✅ 通过: MES 1935279932
✅ 通过: MES 1935279933
✅ 通过: MES 1935279934
✅ 通过: MES 1935280022
✅ 通过: MES 1935280023
✅ 通过: MES 1935280024
🛑 拦截: 触发日损超100，锁定15min
🛑 拦截: MES 处于冷却期至 2026-02-19 23:53:52.388384


Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u5df2\u8fde\u63a5\uff1a usfuture; jfarm; usfarm; fundfarm; ushmds; secdefil. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u672a\u8fde\u63a5\uff1a euhmds.
Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u5df2\u8fde\u63a5\uff1a usfuture; jfarm; usfarm; fundfarm; ushmds; secdefil. \u4ee5\u4e0b\u6570\u636e\u519c\u573a\u672a\u8fde\u63a5\uff1a euhmds.
Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data main

✅ 通过: MES 1366865259
✅ 通过: MES 1366865260
✅ 通过: MES 1366865261
✅ 通过: MES 1366865633
✅ 通过: MES 1366865634
✅ 通过: MES 1366865635


Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u6240\u6709\u6570\u636e\u573a\u5747\u5df2\u8fde\u63a5\uff1a usfuture; usfarm; euhmds; fundfarm; ushmds; secdefil.
Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u6240\u6709\u6570\u636e\u573a\u5747\u5df2\u8fde\u63a5\uff1a usfuture; usfarm; euhmds; fundfarm; ushmds; secdefil.
Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u6240\u6709\u6570\u636e\u573a\u5747\u5df2\u8fde\u63a5\uff1a usfuture; usfarm; euhmds; fundfarm; ushmds; secdefil.
Error 1100, r

👋 系统关闭


Error 1100, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been lost.
Error 1102, reqId -1: Connectivity between %SHORT_COMPNAME% and %SHORT_PRODNAME% has been restored - data maintained. \u6240\u6709\u6570\u636e\u573a\u5747\u5df2\u8fde\u63a5\uff1a usfuture; usfarm; euhmds; fundfarm; ushmds; secdefil.
